In [1]:
from pyspark.sql.functions import col, count, when, year, month, to_date, trim, explode, sequence,lit,coalesce,last,first, round, avg,countDistinct, broadcast
from pyspark.sql import functions as F, Window
from pyspark.sql.types import IntegerType
from pyspark import SparkFiles


# 1. DATA INGESTION (BRONZE LAYER)

# Initial load from raw files into Lakehouse Delta table
# df_raw = spark.read.parquet("Files/bronze/nyc_taxi_2024/*.parquet")
# df_raw.write.format("delta").mode("overwrite").saveAsTable("Taxi")

# Loading tables
df_taxi = spark.read.table("lk_1.dbo.taxi")
df_air = spark.read.table("lk_1.dbo.AirQuality")
df_gdp = spark.read.table("lk_1.dbo.GDP")
df_fx = spark.read.table("lk_1.dbo.FX")


# 2. DATA PROFILING & CLEANING

# --- Air Quality ---
# Profiling: 200+ missing records in the 'value' column
# air_null_counts = df_air.select([
#     count(when(col(c).isNull(), 1)).alias(c)
#     for c in df_air.columns
# ]).first().asDict()

df_air_clean = df_air.na.drop(subset=["value"])
df_air_clean = df_air_clean.withColumn("location_name", trim(col("location_name")))

# --- GDP ---
# Schema standardization: rename dot-notated columns to underscore format
df_gdp_clean = df_gdp.withColumnsRenamed({
    "value.1": "value_1",
    "id.1": "id_1",
    "value.2": "value_2"
})

# Profiling: no missing values identified
# gdp_null_counts = df_gdp.select([
#     count(when(col(c).isNull(), 1)).alias(c)
#     for c in df_gdp.columns
# ]).first().asDict()


# --- Foreign Exchange (FX) ---
# Profiling: 62 missing values identified in the 'OBS_VALUE' column
# fx_null_values = df_fx.select([
#     count(when(col(c).isNull(), 1)).alias(c)
#     for c in df_fx.columns
# ]).first().asDict()
#filtering fx data for 2024 

df_fx_clean = df_fx.filter(
    (col("TIME_PERIOD")>= "2024-01-01") &
    (col("TIME_PERIOD")<= "2024-12-31"))

# --- NYC Taxi ---
# Profiling missing values across all dimensions
# taxi_null_values = df_taxi.select([
#     count(when(col(c).isNull(), 1)).alias(c)
#     for c in df_taxi.columns
# ]).first().asDict()

# Data imputation and attribute cleanup
df_taxi_clean = df_taxi.drop("store_and_fwd_flag").fillna({
    "passenger_count": 1,
    "RatecodeID": 99,
    "congestion_surcharge": 0.0,
    "Airport_fee": 0.0
})
#filtering only profitable trips
df_taxi_clean = df_taxi_clean.filter(
    (col("trip_distance") > 0) &
    (col("fare_amount")  >= 0) & 
    (col("total_amount") > 0))

#cleaning about 50 accidentally-caught rows with incorrect date
df_taxi_clean = (
    df_taxi_clean.filter(
        (col("tpep_pickup_datetime") >= "2024-01-01 00:00:00")
        & (col("tpep_pickup_datetime") < "2025-01-01 00:00:00")
    )
    .withColumn("pickup_date", to_date("tpep_pickup_datetime"))
    .withColumn("year", year("pickup_date"))
    .withColumn("month", month("pickup_date"))
)


#Saving cleaned tables to silver layer
(
    df_taxi_clean.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("delta.enableDeletionVectors", "false")
    .partitionBy("year", "month")
    .saveAsTable("lh_silver.dbo.taxi")
)
    

df_air_clean.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("delta.enableDeletionVectors", "false") \
    .saveAsTable("lh_silver.dbo.air_quality")

df_gdp_clean.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("delta.enableDeletionVectors", "false") \
    .saveAsTable("lh_silver.dbo.gdp")

df_fx_clean.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("delta.enableDeletionVectors", "false") \
    .saveAsTable("lh_silver.dbo.fx")


# Loading boroughs' names
url = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
spark.sparkContext.addFile(url)

df_taxi_zones = (
    spark.read.option("header", "true")
    .csv("file://" + SparkFiles.get("taxi_zone_lookup.csv"))
    .select(
    col("LocationID").cast(IntegerType()).alias("location_id"),
    col("Borough").alias("borough")
    )
    .filter(~col("borough").isin(["Unknown", "EWR"]) & col("borough").isNotNull()))

df_taxi_zones.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("delta.enableDeletionVectors", "false") \
    .saveAsTable("lh_silver.dbo.taxi_zones")

#Defining boroughs for all locations
station_mapping = {
    # Queens
    "Queens": "Queens",
    "Maspeth": "Queens",
    "Bayside, NY": "Queens",
    "56th St & Grand Ave [NYC Testbed Pilot]": "Queens",
    "57th St/Front of School [NYC Testbed Pilot]": "Queens",
    "54th St & Flushing [NYC Testbed Pilot]": "Queens",
    "54th St & Grand Ave [NYC Testbed Pilot]": "Queens",
    "56th St & Flushing [NYC Testbed Pilot]": "Queens",
    "State Dept of Environmental Conservation": "Queens",
    "Hillcrest, NY": "Queens",
    # Brooklyn 
    "Bklyn - PS274": "Brooklyn",
    "Bklyn - PS 314": "Brooklyn",
    "Closest to SA unit on PS9 campus": "Brooklyn",
    "Metal fence on PS9 back campus [NYC Testbed Pilot]": "Brooklyn",
    "Win Son": "Brooklyn",
    "Near Bay 50 St": "Brooklyn",
    "Pfizer Lab": "Brooklyn",
    # Manhattan 
    "CCNY": "Manhattan",
    "Manhattan/IS143": "Manhattan",
    "7th Ave and W 16th St": "Manhattan",
    # Bronx
    "Bronx - IS52": "Bronx",
    "Bronx - IS74": "Bronx",
    "Morrisania": "Bronx",
    # Staten Island
    "Port Richmond": "Staten Island",
}

df_stations = spark.createDataFrame(station_mapping.items(), ["location_name", "borough"])


df_taxi_zones_silver = spark.read.table("lh_silver.dbo.taxi_zones")
df_fx_silver = spark.read.table("lh_silver.dbo.fx")
df_gdp_silver = spark.read.table("lh_silver.dbo.gdp")
df_air_quality_silver = spark.read.table("lh_silver.dbo.air_quality")
df_taxi_silver = spark.read.table("lh_silver.dbo.taxi")

#Creating dimensions
#---DimZone---
df_dim_zone = (
    df_taxi_zones_silver
    .select(col("borough"))
    .distinct()
    )

df_dim_zone.write.format("delta").mode("overwrite").option("overwriteSchema", "true").option("delta.enableDeletionVectors", "false").saveAsTable("lh_gold.dbo.dim_zone")

#---DimDate---
df_dim_date = (
    spark.createDataFrame([{"date": 1}])
    .select(explode(sequence((to_date(lit('2024-01-01'))), to_date(lit('2024-12-31')))).alias("date"))
    .withColumn("year", year("date"))
    .withColumn("month", month("date"))
    )

df_dim_date.write.format("delta").mode("overwrite").option("overwriteSchema", "true").option("delta.enableDeletionVectors", "false").saveAsTable("lh_gold.dbo.dim_date")

#---DimFX---
#adding weekends and holidays
df_fx_full_calendar = (df_dim_date.join(
    df_fx_silver,
    df_fx_silver["TIME_PERIOD"] == df_dim_date["date"],
    "left"
)
.select("date", "OBS_VALUE")
)

#filling missing OBS_VALUEs by (prev+next) / 2
w_prev = Window.orderBy("date").rowsBetween(Window.unboundedPreceding, -1)
w_next = Window.orderBy("date").rowsBetween(1, Window.unboundedFollowing)

df_dim_fx = df_fx_full_calendar.withColumn(
    "OBS_VALUE",
    round(coalesce(
        col("OBS_VALUE"),
        (last(col("OBS_VALUE"), ignorenulls=True).over(w_prev) + first(col("OBS_VALUE"), ignorenulls=True).over(w_next)) / 2,
        first(col("OBS_VALUE"), ignorenulls=True).over(w_next),
        last(col("OBS_VALUE"), ignorenulls=True).over(w_prev)
    ),
    4)
    )
df_dim_fx.write.format("delta").mode("overwrite").option("overwriteSchema", "true").option("delta.enableDeletionVectors", "false").saveAsTable("lh_gold.dbo.dim_fx")

#---DimGDP---
df_dim_gdp = (
    df_gdp_silver
    .filter("date = 2024")
    .select(col("date").cast(IntegerType()).alias("year"), col("value").alias("gdp_value"))
    )
df_dim_gdp.write.format("delta").mode("overwrite").option("overwriteSchema", "true").option("delta.enableDeletionVectors", "false").saveAsTable("lh_gold.dbo.dim_gdp")

#Creating Fact tables
#---FactAir---
target_parameters = ["pm25","o3", "temperature"]
df_fact_air = (
    df_air_quality_silver
    .filter(col("parameter_name").isin(target_parameters))
    .join(broadcast(df_stations),on="location_name", how="inner")
    .join(broadcast(df_dim_date), to_date(df_air_quality_silver["datetime_from_utc"]) == df_dim_date["date"])
    .groupBy("date", "borough").pivot("parameter_name",target_parameters)
    .agg(round(avg("value"), 2))
)
df_fact_air.write.format("delta").mode("overwrite").option("overwriteSchema", "true").option("delta.enableDeletionVectors", "false").saveAsTable("lh_gold.dbo.fact_air_quality_daily")

#counting active days in every month in df_air_quality_silver
# (
#     df_air_quality_silver
#     .withColumn("date", to_date("datetime_from_utc"))
#     .groupBy(year("date").alias("year"), month("date").alias("month"))
#     .agg(
#         countDistinct("date").alias("active_days"),
#         countDistinct("location_name").alias("active_stations")
#     )
#     .orderBy("year", "month")
#     .show(24, truncate=False)
# )


#---FactTaxi---
df_taxi_aggregated = (
    df_taxi_silver
    .select(col("pickup_date").alias("date"),
    col("PULocationID").cast(IntegerType()).alias("location_id"),
    col("total_amount"),
    col("trip_distance")
    )
    .join(broadcast(df_taxi_zones_silver), on="location_id", how="inner")
    .groupBy("date", "borough")
    .agg(
        round(F.sum(col("total_amount")),2).alias("total_revenue"),
        round(F.avg(col("trip_distance")),2).alias("avg_distance"),
        F.count("*").alias("trip_count"))
        )

df_fact_taxi = (
    df_taxi_aggregated
    .join(broadcast(df_dim_date), on="date", how="inner")
    .join(broadcast(df_dim_zone), on="borough", how="inner")
    )
df_fact_taxi.write.format("delta").mode("overwrite").option("overwriteSchema", "true").option("delta.enableDeletionVectors", "false").saveAsTable("lh_gold.dbo.fact_taxi_daily")


StatementMeta(, 4bd37c78-911c-4311-8a22-7f6c6d4a2cd9, 3, Finished, Available, Finished, False)

In [1]:
%%sql
-- Bronze layer (lk_1)
ALTER TABLE lk_1.dbo.taxi SET TBLPROPERTIES ('delta.enableDeletionVectors' = false);
REORG TABLE lk_1.dbo.taxi APPLY (PURGE);

-- Silver layer (lh_silver)
ALTER TABLE lh_silver.dbo.taxi SET TBLPROPERTIES ('delta.enableDeletionVectors' = false);
REORG TABLE lh_silver.dbo.taxi APPLY (PURGE);

-- Gold layer (lh_gold)
ALTER TABLE lh_gold.dbo.fact_taxi_daily SET TBLPROPERTIES ('delta.enableDeletionVectors' = false);
REORG TABLE lh_gold.dbo.fact_taxi_daily APPLY (PURGE);

StatementMeta(, 1754d9a2-63a1-4898-82af-2a500cf6b823, 7, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 1 rows and 2 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 1 rows and 2 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 1 rows and 2 fields>

In [2]:
%%sql
ALTER TABLE lk_1.dbo.taxi DROP FEATURE deletionVectors;
ALTER TABLE lh_silver.dbo.taxi DROP FEATURE deletionVectors;
ALTER TABLE lh_gold.dbo.fact_taxi_daily DROP FEATURE deletionVectors;

StatementMeta(, 1754d9a2-63a1-4898-82af-2a500cf6b823, 10, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>